# Facial PD Detection Model Training
**Based on UFNet (AAAI 2025) — Smile Task**

Architecture: ShallowANN → Linear(n_features → 1) + Dropout + Sigmoid

### Upload these 3 files to Colab:
1. `facial_dataset.csv` — from `UFNet/data/facial_expression_smile/facial_dataset.csv`
2. `dev_set_participants.txt` — from `UFNet/data/dev_set_participants.txt`
3. `test_set_participants.txt` — from `UFNet/data/test_set_participants.txt`

### Output:
- `facial_model.pt` → download and place in `neuroverse-backend/app/ml/models/`
- `facial_scaler.pkl` → download and place in `neuroverse-backend/app/ml/models/`

In [ ]:
!pip install imbalanced-learn -q

In [ ]:
import os
import copy
import random
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

## 1. Upload Data Files
Run this cell, then upload the 3 files when prompted.

In [ ]:
from google.colab import files

print('Upload these 3 files:')
print('  1. facial_dataset.csv')
print('  2. dev_set_participants.txt')
print('  3. test_set_participants.txt')
print()
uploaded = files.upload()

## 2. Configuration (UFNet Best Hyperparameters)

In [ ]:
# From UFNet paper (arxiv 2406.14856) best hyperparameters
SEED = 462
RANDOM_STATE = 154
BATCH_SIZE = 256
NUM_EPOCHS = 64
LEARNING_RATE = 0.03265227174722892
MOMENTUM = 0.5450637936769563
DROPOUT_PROB = 0.10661756438565197
DROP_CORRELATED = False
CORR_THR = 0.95

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Only use smile-related features (from UFNet constants)
FACIAL_EXPRESSIONS = {'smile': True, 'surprise': False, 'disgust': False}

## 3. Load & Prepare Data

In [ ]:
# Load dataset
df = pd.read_csv('facial_dataset.csv')
df.fillna(0, inplace=True)

print(f'Total samples: {len(df)}')
print(f'Columns: {len(df.columns)}')
print(f'PD distribution:\n{df["pd"].value_counts()}')
print()

# Get smile-related feature columns
feature_columns = []
for col in df.columns:
    for expr in FACIAL_EXPRESSIONS:
        if FACIAL_EXPRESSIONS[expr] and expr in col.lower():
            feature_columns.append(col)
            break

print(f'Feature columns ({len(feature_columns)}):')
for c in feature_columns:
    print(f'  {c}')

df_features = df[feature_columns]

In [ ]:
# Optionally drop highly correlated features
if DROP_CORRELATED:
    corr_matrix = df_features.corr()
    drop_cols = set()
    for i in range(len(corr_matrix.columns) - 1):
        for j in range(i + 1):
            if abs(corr_matrix.iloc[j, i + 1]) >= CORR_THR:
                drop_cols.add(corr_matrix.columns[i + 1])
    df.drop(drop_cols, axis=1, inplace=True)
    df_features = df_features.drop(drop_cols, axis=1, errors='ignore')
    print(f'Dropped {len(drop_cols)} correlated features')

features = df_features.to_numpy()

# Labels: yes/Possible/Probable → 1 (PD), no/0 → 0 (Non-PD)
labels = df['pd'].apply(lambda x: 0 if str(x) in ['no', '0'] else 1).to_numpy()
IDs = df['ID']

print(f'\nFeature matrix: {features.shape}')
print(f'PD: {sum(labels)}, Non-PD: {len(labels) - sum(labels)}')

In [ ]:
# Load participant splits
with open('dev_set_participants.txt') as f:
    dev_ids = set(x.strip() for x in f.readlines())
with open('test_set_participants.txt') as f:
    test_ids = set(x.strip() for x in f.readlines())

print(f'Dev IDs: {len(dev_ids)}, Test IDs: {len(test_ids)}')

# Split data
train_f, train_l = [], []
dev_f, dev_l = [], []
test_f, test_l = [], []

for x, l, pid in zip(features, labels, IDs):
    if pid in test_ids:
        test_f.append(x); test_l.append(l)
    elif pid in dev_ids:
        dev_f.append(x); dev_l.append(l)
    else:
        train_f.append(x); train_l.append(l)

X_train, y_train = np.array(train_f), np.array(train_l)
X_dev, y_dev = np.array(dev_f), np.array(dev_l)
X_test, y_test = np.array(test_f), np.array(test_l)

print(f'\nTrain: {len(y_train)} (PD: {sum(y_train)})')
print(f'Dev:   {len(y_dev)} (PD: {sum(y_dev)})')
print(f'Test:  {len(y_test)} (PD: {sum(y_test)})')

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_dev = scaler.transform(X_dev)
X_test = scaler.transform(X_test)
print('Applied StandardScaler')

# SMOTE oversampling for class imbalance
smote = SMOTE(random_state=RANDOM_STATE)
X_train, y_train = smote.fit_resample(X_train, y_train)
print(f'After SMOTE: Train = {len(y_train)} (PD: {sum(y_train)}, Non-PD: {len(y_train)-sum(y_train)})')

## 4. Model Architecture (ShallowANN from UFNet)

In [ ]:
class TensorDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.Tensor(np.asarray(features))
        self.labels = torch.Tensor(labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return len(self.labels)


class ShallowANN(nn.Module):
    """Exact architecture from UFNet unimodal_smile_baal.py"""
    def __init__(self, n_features, drop_prob=0.1):
        super().__init__()
        self.fc = nn.Linear(in_features=n_features, out_features=1, bias=True)
        self.drop = nn.Dropout(p=drop_prob)
        self.sig = nn.Sigmoid()

    def forward(self, x):
        y = self.fc(x)
        y = self.drop(y)
        y = self.sig(y)
        return y


n_features = X_train.shape[1]
model = ShallowANN(n_features, drop_prob=DROPOUT_PROB).to(device)
print(f'ShallowANN: {n_features} input features → 1 output')
print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')

## 5. Training

In [ ]:
# Dataloaders
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(TensorDataset(X_dev, y_dev), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE)

# Optimizer & loss (from UFNet best config)
optimizer = torch.optim.SGD(
    model.parameters(), lr=LEARNING_RATE,
    momentum=MOMENTUM, weight_decay=0.0001
)
criterion = nn.BCELoss()


def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, n_samples = 0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            preds = model(x).reshape(-1)
            loss = criterion(preds, y)
            total_loss += loss.item() * len(y)
            n_samples += len(y)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    labels = np.array(all_labels)
    preds = np.array(all_preds)
    pred_binary = (preds >= 0.5).astype(int)

    metrics = {
        'loss': total_loss / max(n_samples, 1),
        'accuracy': accuracy_score(labels, pred_binary),
        'auroc': roc_auc_score(labels, preds) if len(set(labels)) > 1 else 0.0,
        'f1': f1_score(labels, pred_binary, zero_division=0),
    }
    tn, fp, fn, tp = confusion_matrix(labels, pred_binary, labels=[0, 1]).ravel()
    metrics['sensitivity'] = tp / max(tp + fn, 1)
    metrics['specificity'] = tn / max(tn + fp, 1)
    return metrics

In [ ]:
# Training loop
best_dev_loss = float('inf')
best_model = copy.deepcopy(model)
history = {'train_loss': [], 'dev_loss': [], 'dev_acc': [], 'dev_auroc': []}

print(f'Training for {NUM_EPOCHS} epochs...\n')

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    n_batches = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        preds = model(x).reshape(-1)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1

    avg_train_loss = epoch_loss / max(n_batches, 1)
    dev_metrics = evaluate(model, dev_loader)

    history['train_loss'].append(avg_train_loss)
    history['dev_loss'].append(dev_metrics['loss'])
    history['dev_acc'].append(dev_metrics['accuracy'])
    history['dev_auroc'].append(dev_metrics['auroc'])

    if dev_metrics['loss'] < best_dev_loss:
        best_dev_loss = dev_metrics['loss']
        best_model = copy.deepcopy(model)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS}  '
              f'train_loss={avg_train_loss:.4f}  '
              f'dev_loss={dev_metrics["loss"]:.4f}  '
              f'dev_acc={dev_metrics["accuracy"]:.3f}  '
              f'dev_auroc={dev_metrics["auroc"]:.3f}')

print('\nTraining complete!')

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['dev_loss'], label='Dev')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history['dev_acc'])
axes[1].set_title('Dev Accuracy')

axes[2].plot(history['dev_auroc'])
axes[2].set_title('Dev AUROC')

for ax in axes:
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Test Set Evaluation

In [ ]:
print('='*50)
print('TEST SET EVALUATION (Best Model)')
print('='*50)

test_metrics = evaluate(best_model, test_loader)

for k, v in test_metrics.items():
    print(f'  {k:15s}: {v:.4f}')

# Detailed classification report
best_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = best_model(x).reshape(-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

pred_binary = (np.array(all_preds) >= 0.5).astype(int)
print('\nClassification Report:')
print(classification_report(all_labels, pred_binary, target_names=['Non-PD', 'PD']))

print('Confusion Matrix:')
cm = confusion_matrix(all_labels, pred_binary, labels=[0, 1])
print(f'  TN={cm[0][0]}  FP={cm[0][1]}')
print(f'  FN={cm[1][0]}  TP={cm[1][1]}')

## 8. Save Model & Scaler
Downloads two files:
- **`facial_model.pt`** → place in `neuroverse-backend/app/ml/models/`
- **`facial_scaler.pkl`** → place in `neuroverse-backend/app/ml/models/`

In [ ]:
# Save model weights
torch.save(best_model.cpu().state_dict(), 'facial_model.pt')
print(f'Saved facial_model.pt ({os.path.getsize("facial_model.pt")} bytes)')
print(f'  n_features: {n_features}')
print(f'  feature_columns: {feature_columns}')

# Save scaler
with open('facial_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f'Saved facial_scaler.pkl')

# Save model info for reference
with open('facial_model_info.txt', 'w') as f:
    f.write(f'n_features: {n_features}\n')
    f.write(f'feature_columns: {feature_columns}\n')
    f.write(f'test_metrics: {test_metrics}\n')
    f.write(f'architecture: ShallowANN({n_features} -> 1, dropout={DROPOUT_PROB})\n')
    f.write(f'hyperparameters: lr={LEARNING_RATE}, momentum={MOMENTUM}, epochs={NUM_EPOCHS}, batch={BATCH_SIZE}\n')

# Download files
from google.colab import files
files.download('facial_model.pt')
files.download('facial_scaler.pkl')
files.download('facial_model_info.txt')

print('\n✅ Download these and place in: neuroverse-backend/app/ml/models/')